# 03 - True DragGAN Face Latent Editing

This notebook uses the official DragGAN renderer/optimization loop on the inverted AFHQ Dog latent from notebook `02`.

It is now doing the real thing: face handle points are optimized in StyleGAN latent space. If this step fails, the failure is meaningful: either the generator/inversion/environment is not compatible enough, or the target movement is too far for this latent.

In [ ]:
%matplotlib inline

import json
import math
import os
import subprocess
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "environment.yml").exists() or (candidate / ".git").exists():
            return candidate
    return cwd


PROJECT_ROOT = find_project_root()
PIPELINE_ROOT = PROJECT_ROOT / "data" / "outputs" / "face_draggan_pipeline"
CUTOUT_ROOT = PIPELINE_ROOT / "01_cutout"
INVERSION_ROOT = PIPELINE_ROOT / "02_true_draggan_face_inversion"
DRAGGAN_ROOT = PROJECT_ROOT / "external" / "DragGAN"
CHECKPOINT_DIR = PROJECT_ROOT / "models" / "draggan"
AFHQ_DOG_PKL = CHECKPOINT_DIR / "stylegan2-afhqdog-512x512.pkl"
AFHQ_DOG_URL = "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/afhqdog.pkl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE != "cuda":
    raise RuntimeError("True DragGAN/StyleGAN inversion needs CUDA for practical runtime.")

print("Project root:", PROJECT_ROOT)
print("Pipeline root:", PIPELINE_ROOT)
print("DragGAN repo:", DRAGGAN_ROOT)
print("AFHQ Dog checkpoint:", AFHQ_DOG_PKL)
print("CUDA device:", torch.cuda.get_device_name(0))


In [ ]:
# Large one-shot DragGAN moves often push the latent outside the natural AFHQ-dog face manifold.
# Use staged dragging instead: each stage only moves handles part of the way toward the final target.
DRAG_NUM_STAGES = int(os.environ.get("DOG_TRUE_DRAGGAN_NUM_STAGES", "6"))
MAX_DRAG_STEPS_PER_STAGE = int(os.environ.get("DOG_TRUE_DRAGGAN_STEPS_PER_STAGE", "60"))
DRAG_EARLY_STOP_PX = float(os.environ.get("DOG_TRUE_DRAGGAN_EARLY_STOP_PX", "3.0"))
MAX_DRAG_STEPS = DRAG_NUM_STAGES * MAX_DRAG_STEPS_PER_STAGE
DRAG_LR = float(os.environ.get("DOG_TRUE_DRAGGAN_DRAG_LR", "0.001"))
DRAG_R1 = int(os.environ.get("DOG_TRUE_DRAGGAN_R1", "3"))
DRAG_R2 = int(os.environ.get("DOG_TRUE_DRAGGAN_R2", "12"))
DRAG_LAMBDA_MASK = float(os.environ.get("DOG_TRUE_DRAGGAN_LAMBDA_MASK", "0"))

# Face target modes:
# - nose_drag_with_structure: preferred. Nose is the true DragGAN handle and
#   moves to the original dog's nose; other handles only weakly follow to keep
#   the face from collapsing.
# - yaw_guided_affine: all face handles move in the turn direction,
#   but points on the far/turning side move more than points on the near side.
#   This avoids the unnatural "both eyes collapse toward the center" failure.
# - anisotropic_affine: allows translation/rotation/scale plus a bounded near-far stretch.
# - similarity: conservative fallback; keeps face proportions and only rotates/scales/translates.
# - single_key_rigid_shift: old fallback; every handle receives the same driver displacement.
FACE_TARGET_MODE = os.environ.get("DOG_TRUE_DRAGGAN_FACE_TARGET_MODE", "nose_drag_with_structure")
FACE_DRIVER_KEYPOINT = os.environ.get("DOG_TRUE_DRAGGAN_FACE_DRIVER_KEYPOINT", "nose")
MAX_FACE_HANDLE_DISPLACEMENT_PX = float(os.environ.get("DOG_TRUE_DRAGGAN_MAX_FACE_HANDLE_DISPLACEMENT_PX", "512"))
FACE_AFFINE_BLEND = float(os.environ.get("DOG_TRUE_DRAGGAN_FACE_AFFINE_BLEND", "0.65"))
FACE_AFFINE_MIN_SCALE = float(os.environ.get("DOG_TRUE_DRAGGAN_FACE_AFFINE_MIN_SCALE", "0.82"))
FACE_AFFINE_MAX_SCALE = float(os.environ.get("DOG_TRUE_DRAGGAN_FACE_AFFINE_MAX_SCALE", "1.22"))
FACE_AFFINE_MAX_ANISOTROPY = float(os.environ.get("DOG_TRUE_DRAGGAN_FACE_AFFINE_MAX_ANISOTROPY", "1.18"))
FACE_YAW_GUIDE_BLEND = float(os.environ.get("DOG_TRUE_DRAGGAN_FACE_YAW_GUIDE_BLEND", "0.85"))
FACE_YAW_SIDE_STRENGTH = float(os.environ.get("DOG_TRUE_DRAGGAN_FACE_YAW_SIDE_STRENGTH", "0.38"))
FACE_YAW_MIN_SIDE_WEIGHT = float(os.environ.get("DOG_TRUE_DRAGGAN_FACE_YAW_MIN_SIDE_WEIGHT", "0.60"))
FACE_YAW_MAX_SIDE_WEIGHT = float(os.environ.get("DOG_TRUE_DRAGGAN_FACE_YAW_MAX_SIDE_WEIGHT", "1.42"))
NOSE_DRAG_NOSE_WEIGHT = float(os.environ.get("DOG_TRUE_DRAGGAN_NOSE_DRAG_NOSE_WEIGHT", "1.0"))
NOSE_DRAG_CHIN_WEIGHT = float(os.environ.get("DOG_TRUE_DRAGGAN_NOSE_DRAG_CHIN_WEIGHT", "0.52"))
NOSE_DRAG_EYE_WEIGHT = float(os.environ.get("DOG_TRUE_DRAGGAN_NOSE_DRAG_EYE_WEIGHT", "0.28"))
NOSE_DRAG_EAR_BASE_WEIGHT = float(os.environ.get("DOG_TRUE_DRAGGAN_NOSE_DRAG_EAR_BASE_WEIGHT", "0.18"))
NOSE_DRAG_STRUCTURE_AFFINE_BLEND = float(os.environ.get("DOG_TRUE_DRAGGAN_NOSE_DRAG_STRUCTURE_AFFINE_BLEND", "0.18"))
NOSE_DRAG_MAX_SUPPORT_DISPLACEMENT_PX = float(os.environ.get("DOG_TRUE_DRAGGAN_NOSE_DRAG_MAX_SUPPORT_DISPLACEMENT_PX", "80"))

OUTPUT_ROOT = PIPELINE_ROOT / "03_true_draggan_face_edit"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Drag stages:", DRAG_NUM_STAGES)
print("Max drag steps per stage:", MAX_DRAG_STEPS_PER_STAGE)
print("Total max drag steps:", MAX_DRAG_STEPS)
print("Drag early-stop px:", DRAG_EARLY_STOP_PX)
print("Drag LR:", DRAG_LR)
print("Face target mode:", FACE_TARGET_MODE)
print("Face driver keypoint:", FACE_DRIVER_KEYPOINT)
print("Max face handle displacement px:", MAX_FACE_HANDLE_DISPLACEMENT_PX)
print("Face affine blend:", FACE_AFFINE_BLEND)
print("Face affine scale range:", (FACE_AFFINE_MIN_SCALE, FACE_AFFINE_MAX_SCALE))
print("Face affine max anisotropy:", FACE_AFFINE_MAX_ANISOTROPY)
print("Face yaw guide blend:", FACE_YAW_GUIDE_BLEND)
print("Face yaw side strength:", FACE_YAW_SIDE_STRENGTH)
print("Face yaw side weight range:", (FACE_YAW_MIN_SIDE_WEIGHT, FACE_YAW_MAX_SIDE_WEIGHT))
print("Nose drag weights:", {
    "nose": NOSE_DRAG_NOSE_WEIGHT,
    "chin": NOSE_DRAG_CHIN_WEIGHT,
    "eye": NOSE_DRAG_EYE_WEIGHT,
    "ear_base": NOSE_DRAG_EAR_BASE_WEIGHT,
})
print("Nose drag structure affine blend:", NOSE_DRAG_STRUCTURE_AFFINE_BLEND)
print("Nose drag max support displacement px:", NOSE_DRAG_MAX_SUPPORT_DISPLACEMENT_PX)
print("Output root:", OUTPUT_ROOT)

In [ ]:

def ensure_draggan_repo():
    if DRAGGAN_ROOT.exists():
        return
    DRAGGAN_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/XingangPan/DragGAN.git",
        str(DRAGGAN_ROOT),
    ])


def ensure_afhq_dog_checkpoint():
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    if AFHQ_DOG_PKL.exists():
        return
    import requests
    from tqdm.auto import tqdm

    tmp = AFHQ_DOG_PKL.with_suffix(".part")
    with requests.get(AFHQ_DOG_URL, stream=True, timeout=30) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(tmp, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc="Downloading AFHQ Dog StyleGAN2-ADA") as pbar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    tmp.replace(AFHQ_DOG_PKL)


def add_draggan_to_path():
    ensure_draggan_repo()
    draggan_str = str(DRAGGAN_ROOT)
    if draggan_str not in sys.path:
        sys.path.insert(0, draggan_str)



def force_stylegan_reference_ops():
    """Force StyleGAN/DragGAN to use PyTorch reference ops instead of compiling CUDA plugins.

    The official StyleGAN code tries to build custom CUDA extensions such as
    `bias_act_plugin` and `upfirdn2d_plugin`. On Windows this requires MSVC
    Build Tools. For this notebook branch we prefer reliability over speed, so
    these wrappers route calls to the built-in reference implementations.
    """
    from torch_utils.ops import bias_act, filtered_lrelu, upfirdn2d

    def bias_act_ref(x, b=None, dim=1, act="linear", alpha=None, gain=None, clamp=None, impl="ref"):
        return bias_act._bias_act_ref(x=x, b=b, dim=dim, act=act, alpha=alpha, gain=gain, clamp=clamp)

    def upfirdn2d_ref(x, f, up=1, down=1, padding=0, flip_filter=False, gain=1, impl="ref"):
        return upfirdn2d._upfirdn2d_ref(x=x, f=f, up=up, down=down, padding=padding, flip_filter=flip_filter, gain=gain)

    def filtered_lrelu_ref(x, fu=None, fd=None, b=None, up=1, down=1, padding=0, gain=np.sqrt(2), slope=0.2, clamp=None, flip_filter=False, impl="ref"):
        return filtered_lrelu._filtered_lrelu_ref(
            x=x,
            fu=fu,
            fd=fd,
            b=b,
            up=up,
            down=down,
            padding=padding,
            gain=gain,
            slope=slope,
            clamp=clamp,
            flip_filter=flip_filter,
        )

    bias_act.bias_act = bias_act_ref
    upfirdn2d.upfirdn2d = upfirdn2d_ref
    filtered_lrelu.filtered_lrelu = filtered_lrelu_ref
    print("StyleGAN custom CUDA plugins disabled: using PyTorch reference ops.")


def read_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def read_rgba(path):
    return np.array(Image.open(path).convert("RGBA"))


def save_rgb(path, image):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(np.clip(image, 0, 255).astype(np.uint8)).save(path)


def show_images(items, cols=3, figsize=(16, 8)):
    rows = math.ceil(len(items) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for ax, (title, image) in zip(axes, items):
        ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")
    for ax in axes[len(items):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def load_active_cutout_jobs(max_pairs):
    manifest = CUTOUT_ROOT / "_active_batch_manifest.json"
    if not manifest.exists():
        manifest = CUTOUT_ROOT / "batch_manifest.json"
    if not manifest.exists():
        raise FileNotFoundError(f"No cutout manifest found at {CUTOUT_ROOT}. Run notebook 01 first.")
    with open(manifest, "r", encoding="utf-8") as f:
        jobs = json.load(f)
    if max_pairs is not None:
        jobs = jobs[: int(max_pairs)]
    return jobs


DOG_KEYPOINT_NAMES = [
    "front_left_paw", "front_left_knee", "front_left_elbow",
    "rear_left_paw", "rear_left_knee", "rear_left_elbow",
    "front_right_paw", "front_right_knee", "front_right_elbow",
    "rear_right_paw", "rear_right_knee", "rear_right_elbow",
    "tail_start", "tail_end",
    "left_ear_base", "right_ear_base",
    "nose", "chin",
    "left_ear_tip", "right_ear_tip",
    "left_eye", "right_eye",
    "withers", "throat",
]
FACE_NAMES = ["nose", "chin", "left_eye", "right_eye", "left_ear_base", "right_ear_base", "left_ear_tip", "right_ear_tip", "throat"]


def deserialize_pose_record(record):
    if record is None:
        return None
    out = dict(record)
    out["points"] = np.asarray(record["points"], dtype=np.float32)
    out["visible"] = np.asarray(record["visible"], dtype=bool)
    return out


def face_keypoints(pose):
    points = {}
    for name in FACE_NAMES:
        idx = DOG_KEYPOINT_NAMES.index(name)
        if idx < len(pose["points"]) and bool(pose["visible"][idx]):
            points[name] = pose["points"][idx].astype(np.float32)
    return points


def square_crop_box_from_points(points, image_shape, scale=1.80, min_side=96):
    h, w = image_shape[:2]
    pts = np.asarray(list(points.values()), dtype=np.float32)
    if len(pts) < 3:
        raise RuntimeError("Too few visible face keypoints for true DragGAN face crop.")
    x1, y1 = pts.min(axis=0)
    x2, y2 = pts.max(axis=0)
    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    side = max(float(x2 - x1), float(y2 - y1), float(min_side)) * float(scale)
    x1 = int(round(cx - side / 2.0))
    y1 = int(round(cy - side / 2.0))
    x2 = int(round(cx + side / 2.0))
    y2 = int(round(cy + side / 2.0))
    pad_left = max(0, -x1)
    pad_top = max(0, -y1)
    pad_right = max(0, x2 - w)
    pad_bottom = max(0, y2 - h)
    x1c, y1c = max(0, x1), max(0, y1)
    x2c, y2c = min(w, x2), min(h, y2)
    return [x1, y1, x2, y2], [x1c, y1c, x2c, y2c], [pad_left, pad_top, pad_right, pad_bottom]


def crop_square_with_padding(image, raw_box, clipped_box, padding, fill=(127, 127, 127, 0)):
    x1, y1, x2, y2 = raw_box
    x1c, y1c, x2c, y2c = clipped_box
    side = max(1, x2 - x1, y2 - y1)
    channels = image.shape[2] if image.ndim == 3 else 1
    if channels == 4:
        canvas = np.zeros((side, side, 4), dtype=np.uint8)
        canvas[..., :] = np.array(fill, dtype=np.uint8)
    else:
        canvas = np.zeros((side, side, channels), dtype=np.uint8)
        canvas[..., :] = np.array(fill[:channels], dtype=np.uint8)
    dx = x1c - x1
    dy = y1c - y1
    canvas[dy:dy + (y2c - y1c), dx:dx + (x2c - x1c)] = image[y1c:y2c, x1c:x2c]
    return canvas


def points_to_crop512(points, raw_box):
    x1, y1, x2, y2 = raw_box
    side = max(1, x2 - x1, y2 - y1)
    out = {}
    for name, pt in points.items():
        out[name] = [
            float((pt[0] - x1) / side * 512.0),
            float((pt[1] - y1) / side * 512.0),
        ]
    return out


def draw_points(image, points, color=(255, 60, 60)):
    out = image.copy()
    for name, (x, y) in points.items():
        cv2.circle(out, (int(round(x)), int(round(y))), 5, color, -1)
        cv2.putText(out, name, (int(round(x)) + 6, int(round(y)) - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.38, color, 1, cv2.LINE_AA)
    return out


In [ ]:
ensure_draggan_repo()
ensure_afhq_dog_checkpoint()
add_draggan_to_path()
force_stylegan_reference_ops()

import dnnlib
from viz.renderer import Renderer

manifest_path = INVERSION_ROOT / "batch_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(f"No inversion manifest found at {manifest_path}. Run notebook 02 first.")
with open(manifest_path, "r", encoding="utf-8") as f:
    inversion_jobs = json.load(f)

print("Inversion jobs:", len(inversion_jobs))

In [ ]:
def render_uint8_from_renderer(renderer, res):
    arr = renderer.to_cpu(res.image).detach().cpu().numpy()
    return np.asarray(arr, dtype=np.uint8)


def point_xy(points, name):
    if name not in points:
        return None
    return np.asarray(points[name], dtype=np.float32)


def cap_vector(delta_xy, max_distance):
    delta = np.asarray(delta_xy, dtype=np.float32)
    dist = float(np.linalg.norm(delta))
    if dist <= max_distance or dist < 1e-6:
        return delta, False, dist
    return delta / dist * float(max_distance), True, dist


def build_single_key_rigid_targets(rep_points, org_points, handle_names):
    rep_driver = point_xy(rep_points, FACE_DRIVER_KEYPOINT)
    org_driver = point_xy(org_points, FACE_DRIVER_KEYPOINT)
    if rep_driver is None or org_driver is None:
        raise RuntimeError(f"Driver keypoint '{FACE_DRIVER_KEYPOINT}' must exist on both faces.")

    raw_delta = org_driver - rep_driver
    delta, was_capped, raw_distance = cap_vector(raw_delta, MAX_FACE_HANDLE_DISPLACEMENT_PX)
    targets = {}
    for name in handle_names:
        pt = point_xy(rep_points, name)
        if pt is not None:
            # All handles share exactly the same target displacement. The target
            # geometry is therefore a translated copy of the replacement geometry,
            # not a reshaped copy of the original dog's face.
            targets[name] = pt + delta

    transform_meta = {
        "mode": "single_key_rigid_shift",
        "driver_keypoint": FACE_DRIVER_KEYPOINT,
        "driver_raw_delta_xy": raw_delta.tolist(),
        "driver_used_delta_xy": delta.tolist(),
        "driver_raw_distance_px": float(raw_distance),
        "driver_delta_capped": bool(was_capped),
        "translation_applied": True,
        "rotation_applied": False,
        "max_face_handle_displacement_px": float(MAX_FACE_HANDLE_DISPLACEMENT_PX),
        "rep_driver_xy": rep_driver.tolist(),
        "org_driver_xy": org_driver.tolist(),
    }
    return targets, transform_meta


def _points_xy_for_names(rep_points, org_points, handle_names):
    src = []
    dst = []
    used_names = []
    for name in handle_names:
        rep_pt = point_xy(rep_points, name)
        org_pt = point_xy(org_points, name)
        if rep_pt is None or org_pt is None:
            continue
        src.append(rep_pt)
        dst.append(org_pt)
        used_names.append(name)
    if len(src) < 2:
        raise RuntimeError("Too few shared stable face handles for affine-guided dragging.")
    return np.asarray(src, dtype=np.float32), np.asarray(dst, dtype=np.float32), used_names


def _apply_affine_2x3(matrix, points_xy):
    pts = np.asarray(points_xy, dtype=np.float32)
    ones = np.ones((len(pts), 1), dtype=np.float32)
    return np.concatenate([pts, ones], axis=1) @ np.asarray(matrix, dtype=np.float32).T


def _identity_affine():
    return np.asarray([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]], dtype=np.float32)


def estimate_similarity_affine(src_xy, dst_xy):
    if len(src_xy) < 2:
        raise RuntimeError("Similarity transform needs at least two shared handles.")
    matrix, inliers = cv2.estimateAffinePartial2D(src_xy, dst_xy, method=cv2.LMEDS)
    if matrix is None:
        matrix = _identity_affine()
        matrix[:, 2] = dst_xy.mean(axis=0) - src_xy.mean(axis=0)
        inliers = np.ones((len(src_xy), 1), dtype=np.uint8)
    return matrix.astype(np.float32), None if inliers is None else inliers.reshape(-1).astype(bool)


def estimate_full_affine(src_xy, dst_xy, fallback_matrix):
    if len(src_xy) < 3:
        return fallback_matrix.copy(), np.zeros((len(src_xy),), dtype=bool), "not_enough_points"
    matrix, inliers = cv2.estimateAffine2D(src_xy, dst_xy, method=cv2.LMEDS)
    if matrix is None:
        return fallback_matrix.copy(), np.zeros((len(src_xy),), dtype=bool), "estimate_failed"
    return matrix.astype(np.float32), None if inliers is None else inliers.reshape(-1).astype(bool), "estimated"


def regularize_affine_matrix(matrix):
    """Bound affine scale/anisotropy so near-far perspective does not destroy identity."""
    mat = np.asarray(matrix, dtype=np.float32).copy()
    linear = mat[:, :2]
    trans = mat[:, 2].copy()
    try:
        u, singular, vt = np.linalg.svd(linear)
    except np.linalg.LinAlgError:
        return mat, {"regularization_failed": True}

    singular = np.asarray(singular, dtype=np.float32)
    original_singular = singular.copy()
    mean_scale = float(np.sqrt(max(1e-6, singular[0] * singular[1])))
    mean_scale = float(np.clip(mean_scale, FACE_AFFINE_MIN_SCALE, FACE_AFFINE_MAX_SCALE))
    max_ratio = max(1.0, float(FACE_AFFINE_MAX_ANISOTROPY))
    ratio = float(max(singular) / max(1e-6, min(singular)))
    if ratio > max_ratio:
        singular = np.asarray([mean_scale * np.sqrt(max_ratio), mean_scale / np.sqrt(max_ratio)], dtype=np.float32)
    else:
        singular = np.clip(singular, FACE_AFFINE_MIN_SCALE, FACE_AFFINE_MAX_SCALE)

    mat[:, :2] = (u @ np.diag(singular) @ vt).astype(np.float32)
    mat[:, 2] = trans
    return mat, {
        "original_singular_values": original_singular.tolist(),
        "regularized_singular_values": singular.tolist(),
        "original_anisotropy_ratio": ratio,
        "max_anisotropy_ratio": float(max_ratio),
        "scale_range": [float(FACE_AFFINE_MIN_SCALE), float(FACE_AFFINE_MAX_SCALE)],
    }


def build_similarity_or_affine_targets(rep_points, org_points, handle_names, mode):
    src_xy, dst_xy, used_names = _points_xy_for_names(rep_points, org_points, handle_names)
    similarity_matrix, similarity_inliers = estimate_similarity_affine(src_xy, dst_xy)
    similarity_targets = _apply_affine_2x3(similarity_matrix, src_xy)

    if mode == "similarity":
        final_targets = similarity_targets
        used_matrix = similarity_matrix
        affine_matrix = None
        affine_status = "disabled"
        regularization_meta = None
        blend = 0.0
    elif mode == "anisotropic_affine":
        affine_matrix, affine_inliers, affine_status = estimate_full_affine(src_xy, dst_xy, similarity_matrix)
        affine_matrix, regularization_meta = regularize_affine_matrix(affine_matrix)
        affine_targets = _apply_affine_2x3(affine_matrix, src_xy)
        blend = float(np.clip(FACE_AFFINE_BLEND, 0.0, 1.0))
        final_targets = similarity_targets * (1.0 - blend) + affine_targets * blend
        used_matrix = similarity_matrix * (1.0 - blend) + affine_matrix * blend
    else:
        raise ValueError(f"Unsupported affine face mode: {mode}")

    targets = {}
    displacement_records = {}
    for name, src_pt, target_pt in zip(used_names, src_xy, final_targets):
        delta, was_capped, raw_distance = cap_vector(target_pt - src_pt, MAX_FACE_HANDLE_DISPLACEMENT_PX)
        targets[name] = src_pt + delta
        displacement_records[name] = {
            "raw_target_xy": target_pt.tolist(),
            "used_target_xy": targets[name].tolist(),
            "raw_distance_px": float(raw_distance),
            "capped": bool(was_capped),
        }

    transform_meta = {
        "mode": mode,
        "driver_keypoint": FACE_DRIVER_KEYPOINT,
        "handle_names_used_for_fit": used_names,
        "similarity_matrix_2x3": similarity_matrix.tolist(),
        "similarity_inliers": None if similarity_inliers is None else similarity_inliers.astype(bool).tolist(),
        "affine_matrix_2x3": None if affine_matrix is None else affine_matrix.tolist(),
        "used_matrix_2x3": used_matrix.tolist(),
        "affine_status": affine_status,
        "affine_blend": float(blend),
        "affine_regularization": regularization_meta,
        "max_face_handle_displacement_px": float(MAX_FACE_HANDLE_DISPLACEMENT_PX),
        "per_handle_displacement": displacement_records,
    }
    return targets, transform_meta


def estimate_turn_sign(rep_points, org_points):
    """Estimate image-space turn direction from stable driver geometry.

    Positive means the face should move/turn to image-right; negative means
    image-left. Prefer the nose driver. If the nose barely moves, fall back to
    the change of nose offset from the eye/ear center.
    """
    rep_driver = point_xy(rep_points, FACE_DRIVER_KEYPOINT)
    org_driver = point_xy(org_points, FACE_DRIVER_KEYPOINT)
    if rep_driver is not None and org_driver is not None:
        dx = float(org_driver[0] - rep_driver[0])
        if abs(dx) >= 2.0:
            return float(np.sign(dx)), dx, "driver_dx"

    stable_names = ["left_eye", "right_eye", "left_ear_base", "right_ear_base"]
    rep_side = [point_xy(rep_points, name) for name in stable_names if point_xy(rep_points, name) is not None]
    org_side = [point_xy(org_points, name) for name in stable_names if point_xy(org_points, name) is not None]
    if rep_driver is not None and org_driver is not None and rep_side and org_side:
        rep_center = np.asarray(rep_side, dtype=np.float32).mean(axis=0)
        org_center = np.asarray(org_side, dtype=np.float32).mean(axis=0)
        dx = float((org_driver[0] - org_center[0]) - (rep_driver[0] - rep_center[0]))
        if abs(dx) >= 1.0:
            return float(np.sign(dx)), dx, "driver_relative_to_face_center"
    return 0.0, 0.0, "undetermined"


def build_yaw_guided_affine_targets(rep_points, org_points, handle_names):
    src_xy, dst_xy, used_names = _points_xy_for_names(rep_points, org_points, handle_names)
    similarity_matrix, similarity_inliers = estimate_similarity_affine(src_xy, dst_xy)
    similarity_targets = _apply_affine_2x3(similarity_matrix, src_xy)

    affine_matrix, affine_inliers, affine_status = estimate_full_affine(src_xy, dst_xy, similarity_matrix)
    affine_matrix, regularization_meta = regularize_affine_matrix(affine_matrix)
    affine_targets = _apply_affine_2x3(affine_matrix, src_xy)
    affine_blend = float(np.clip(FACE_AFFINE_BLEND, 0.0, 1.0))
    affine_reference_targets = similarity_targets * (1.0 - affine_blend) + affine_targets * affine_blend

    rep_driver = point_xy(rep_points, FACE_DRIVER_KEYPOINT)
    org_driver = point_xy(org_points, FACE_DRIVER_KEYPOINT)
    if rep_driver is None or org_driver is None:
        raise RuntimeError(f"Driver keypoint '{FACE_DRIVER_KEYPOINT}' is required for yaw-guided face dragging.")
    driver_delta, driver_was_capped, driver_raw_distance = cap_vector(org_driver - rep_driver, MAX_FACE_HANDLE_DISPLACEMENT_PX)
    turn_sign, turn_signal, turn_source = estimate_turn_sign(rep_points, org_points)
    if turn_sign == 0.0 and abs(float(driver_delta[0])) >= 1.0:
        turn_sign = float(np.sign(driver_delta[0]))

    nose = point_xy(rep_points, "nose")
    if nose is None:
        nose = src_xy.mean(axis=0)
    face_width = max(32.0, float(np.percentile(src_xy[:, 0], 90) - np.percentile(src_xy[:, 0], 10)))
    yaw_blend = float(np.clip(FACE_YAW_GUIDE_BLEND, 0.0, 1.0))
    side_strength = float(FACE_YAW_SIDE_STRENGTH)
    min_weight = float(FACE_YAW_MIN_SIDE_WEIGHT)
    max_weight = float(FACE_YAW_MAX_SIDE_WEIGHT)

    targets = {}
    displacement_records = {}
    side_records = {}
    for name, src_pt, affine_ref_target in zip(used_names, src_xy, affine_reference_targets):
        affine_delta = affine_ref_target - src_pt
        rel_x = float(np.clip((src_pt[0] - nose[0]) / face_width, -1.0, 1.0))
        side_weight = float(np.clip(1.0 - turn_sign * side_strength * rel_x, min_weight, max_weight))

        # The horizontal component follows a yaw-like trend: all handles move in
        # the turn direction, but the far/turning side moves more. This avoids
        # left/right eye handles being pulled toward each other by noisy target
        # landmark positions.
        guided_dx = float(driver_delta[0]) * side_weight
        if turn_sign != 0.0 and abs(guided_dx) >= 1.0 and np.sign(guided_dx) != turn_sign:
            guided_dx = abs(guided_dx) * turn_sign
        final_dx = float(affine_delta[0]) * (1.0 - yaw_blend) + guided_dx * yaw_blend
        if turn_sign != 0.0 and abs(driver_delta[0]) >= 2.0 and final_dx * turn_sign < 0:
            final_dx = float(driver_delta[0]) * min_weight

        final_delta = np.asarray([final_dx, float(affine_delta[1])], dtype=np.float32)
        used_delta, was_capped, raw_distance = cap_vector(final_delta, MAX_FACE_HANDLE_DISPLACEMENT_PX)
        targets[name] = src_pt + used_delta
        displacement_records[name] = {
            "affine_reference_target_xy": affine_ref_target.tolist(),
            "used_target_xy": targets[name].tolist(),
            "raw_delta_xy": final_delta.tolist(),
            "used_delta_xy": used_delta.tolist(),
            "raw_distance_px": float(raw_distance),
            "capped": bool(was_capped),
        }
        side_records[name] = {"relative_x_from_nose": rel_x, "side_weight": side_weight}

    transform_meta = {
        "mode": "yaw_guided_affine",
        "driver_keypoint": FACE_DRIVER_KEYPOINT,
        "driver_raw_delta_xy": (org_driver - rep_driver).tolist(),
        "driver_used_delta_xy": driver_delta.tolist(),
        "driver_raw_distance_px": float(driver_raw_distance),
        "driver_delta_capped": bool(driver_was_capped),
        "turn_sign_x": float(turn_sign),
        "turn_signal": float(turn_signal),
        "turn_source": turn_source,
        "yaw_guide_blend": float(yaw_blend),
        "yaw_side_strength": float(side_strength),
        "yaw_side_weight_range": [min_weight, max_weight],
        "face_width_for_side_weight": float(face_width),
        "handle_names_used_for_fit": used_names,
        "similarity_matrix_2x3": similarity_matrix.tolist(),
        "similarity_inliers": None if similarity_inliers is None else similarity_inliers.astype(bool).tolist(),
        "affine_matrix_2x3": affine_matrix.tolist(),
        "affine_status": affine_status,
        "affine_blend": float(affine_blend),
        "affine_regularization": regularization_meta,
        "side_weight_by_handle": side_records,
        "max_face_handle_displacement_px": float(MAX_FACE_HANDLE_DISPLACEMENT_PX),
        "per_handle_displacement": displacement_records,
    }
    return targets, transform_meta


def nose_structure_weight(name):
    if name == FACE_DRIVER_KEYPOINT:
        return float(NOSE_DRAG_NOSE_WEIGHT)
    if name == "chin":
        return float(NOSE_DRAG_CHIN_WEIGHT)
    if name in {"left_eye", "right_eye"}:
        return float(NOSE_DRAG_EYE_WEIGHT)
    if name in {"left_ear_base", "right_ear_base"}:
        return float(NOSE_DRAG_EAR_BASE_WEIGHT)
    return 0.20


def build_nose_drag_with_structure_targets(rep_points, org_points, handle_names):
    rep_driver = point_xy(rep_points, FACE_DRIVER_KEYPOINT)
    org_driver = point_xy(org_points, FACE_DRIVER_KEYPOINT)
    if rep_driver is None or org_driver is None:
        raise RuntimeError(f"Driver keypoint '{FACE_DRIVER_KEYPOINT}' must exist on both faces.")

    src_xy, dst_xy, used_names = _points_xy_for_names(rep_points, org_points, handle_names)
    similarity_matrix, similarity_inliers = estimate_similarity_affine(src_xy, dst_xy)
    similarity_targets = _apply_affine_2x3(similarity_matrix, src_xy)

    raw_driver_delta = org_driver - rep_driver
    driver_delta, driver_was_capped, driver_raw_distance = cap_vector(raw_driver_delta, MAX_FACE_HANDLE_DISPLACEMENT_PX)
    structure_blend = float(np.clip(NOSE_DRAG_STRUCTURE_AFFINE_BLEND, 0.0, 1.0))
    support_cap = float(NOSE_DRAG_MAX_SUPPORT_DISPLACEMENT_PX)

    targets = {}
    displacement_records = {}
    for name, src_pt, sim_target in zip(used_names, src_xy, similarity_targets):
        if name == FACE_DRIVER_KEYPOINT:
            target_delta = driver_delta * float(NOSE_DRAG_NOSE_WEIGHT)
            max_distance = MAX_FACE_HANDLE_DISPLACEMENT_PX
        else:
            follow_weight = nose_structure_weight(name)
            # Support handles move in the same broad nose direction, but much
            # less. A small similarity residual keeps the face from becoming a
            # pure rubber-sheet translation while avoiding direct per-landmark
            # chasing of the original dog's face.
            follow_delta = driver_delta * follow_weight
            structure_delta = sim_target - src_pt
            target_delta = follow_delta * (1.0 - structure_blend) + structure_delta * structure_blend
            max_distance = support_cap

        used_delta, was_capped, raw_distance = cap_vector(target_delta, max_distance)
        targets[name] = src_pt + used_delta
        displacement_records[name] = {
            "follow_weight": float(nose_structure_weight(name)),
            "similarity_reference_target_xy": sim_target.tolist(),
            "raw_delta_xy": target_delta.tolist(),
            "used_delta_xy": used_delta.tolist(),
            "used_target_xy": targets[name].tolist(),
            "raw_distance_px": float(raw_distance),
            "capped": bool(was_capped),
            "max_distance_px": float(max_distance),
        }

    transform_meta = {
        "mode": "nose_drag_with_structure",
        "driver_keypoint": FACE_DRIVER_KEYPOINT,
        "driver_raw_delta_xy": raw_driver_delta.tolist(),
        "driver_used_delta_xy": driver_delta.tolist(),
        "driver_raw_distance_px": float(driver_raw_distance),
        "driver_delta_capped": bool(driver_was_capped),
        "nose_weight": float(NOSE_DRAG_NOSE_WEIGHT),
        "chin_weight": float(NOSE_DRAG_CHIN_WEIGHT),
        "eye_weight": float(NOSE_DRAG_EYE_WEIGHT),
        "ear_base_weight": float(NOSE_DRAG_EAR_BASE_WEIGHT),
        "structure_affine_blend": float(structure_blend),
        "max_support_displacement_px": float(support_cap),
        "handle_names_used_for_fit": used_names,
        "similarity_matrix_2x3": similarity_matrix.tolist(),
        "similarity_inliers": None if similarity_inliers is None else similarity_inliers.astype(bool).tolist(),
        "max_face_handle_displacement_px": float(MAX_FACE_HANDLE_DISPLACEMENT_PX),
        "per_handle_displacement": displacement_records,
    }
    return targets, transform_meta


def choose_handle_pairs(meta):
    rep_points = meta["replacement_face_points_512"]
    org_points = meta["original_face_points_512"]
    stable_face_handles = {"nose", "chin", "left_eye", "right_eye", "left_ear_base", "right_ear_base"}
    names = [name for name in meta["shared_face_names"] if name in stable_face_handles]
    if FACE_DRIVER_KEYPOINT not in names:
        raise RuntimeError(f"Driver keypoint '{FACE_DRIVER_KEYPOINT}' is required for rigid face dragging.")
    if len(names) < 2:
        raise RuntimeError("Too few shared face handles for true DragGAN.")

    if FACE_TARGET_MODE == "single_key_rigid_shift":
        target_points, transform_meta = build_single_key_rigid_targets(rep_points, org_points, names)
    elif FACE_TARGET_MODE == "nose_drag_with_structure":
        target_points, transform_meta = build_nose_drag_with_structure_targets(rep_points, org_points, names)
    elif FACE_TARGET_MODE == "yaw_guided_affine":
        target_points, transform_meta = build_yaw_guided_affine_targets(rep_points, org_points, names)
    elif FACE_TARGET_MODE in {"similarity", "anisotropic_affine"}:
        target_points, transform_meta = build_similarity_or_affine_targets(rep_points, org_points, names, FACE_TARGET_MODE)
    else:
        raise ValueError(f"Unsupported FACE_TARGET_MODE: {FACE_TARGET_MODE}")
    names = [name for name in names if name in target_points]
    starts_xy = [rep_points[name] for name in names]
    targets_xy = [target_points[name].tolist() for name in names]

    # DragGAN renderer expects [y, x], not [x, y].
    starts_yx = [[int(round(p[1])), int(round(p[0]))] for p in starts_xy]
    targets_yx = [[int(round(p[1])), int(round(p[0]))] for p in targets_xy]
    return names, starts_yx, targets_yx, transform_meta


def draw_handle_plan(image, starts_yx, targets_yx, names):
    out = image.copy()
    for name, start, target in zip(names, starts_yx, targets_yx):
        sy, sx = int(round(start[0])), int(round(start[1]))
        ty, tx = int(round(target[0])), int(round(target[1]))
        cv2.circle(out, (sx, sy), 5, (80, 220, 255), -1)
        cv2.circle(out, (tx, ty), 5, (255, 90, 90), -1)
        cv2.arrowedLine(out, (sx, sy), (tx, ty), (255, 210, 40), 2, cv2.LINE_AA, tipLength=0.25)
        cv2.putText(out, name, (sx + 6, sy - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.36, (80, 220, 255), 1, cv2.LINE_AA)
    return out


def interpolate_stage_targets(starts_yx, targets_yx, stage_index, num_stages):
    starts = np.asarray(starts_yx, dtype=np.float32)
    targets = np.asarray(targets_yx, dtype=np.float32)
    fraction = float(stage_index) / float(max(1, num_stages))
    stage_targets = starts + (targets - starts) * fraction
    return [[int(round(p[0])), int(round(p[1]))] for p in stage_targets]


def resolve_generator_pkl(meta):
    pkl = Path(meta.get("generator_pkl_for_draggan", meta.get("afhq_dog_pkl", str(AFHQ_DOG_PKL))))
    if not pkl.is_absolute():
        pkl = PROJECT_ROOT / pkl
    if not pkl.exists():
        raise FileNotFoundError(f"DragGAN generator checkpoint not found: {pkl}. Re-run notebook 02.")
    return pkl


def run_draggan(w_plus, starts_yx, targets_yx, generator_pkl):
    renderer = Renderer(disable_timing=True)
    res = dnnlib.EasyDict()
    w = torch.from_numpy(w_plus).to(DEVICE)
    renderer.init_network(
        res,
        pkl=str(generator_pkl),
        w_load=w,
        w_plus=True,
        reset_w=True,
        lr=DRAG_LR,
        trunc_psi=0.7,
    )
    renderer._render_drag_impl(res, points=[], targets=[], is_drag=False, to_pil=False)
    start_img = render_uint8_from_renderer(renderer, res)

    current_points = [[int(round(p[0])), int(round(p[1]))] for p in starts_yx]
    drag_mask = None
    history = []
    global_step = 0
    for stage_index in range(1, DRAG_NUM_STAGES + 1):
        stage_targets = interpolate_stage_targets(starts_yx, targets_yx, stage_index, DRAG_NUM_STAGES)
        if hasattr(res, "stop"):
            res.stop = False
        print(f"stage {stage_index}/{DRAG_NUM_STAGES} | target fraction={stage_index / max(1, DRAG_NUM_STAGES):.3f}")
        for step_in_stage in range(MAX_DRAG_STEPS_PER_STAGE):
            renderer._render_drag_impl(
                res,
                points=current_points,
                targets=stage_targets,
                mask=drag_mask,
                lambda_mask=DRAG_LAMBDA_MASK,
                reg=0,
                feature_idx=5,
                r1=DRAG_R1,
                r2=DRAG_R2,
                trunc_psi=0.7,
                is_drag=True,
                to_pil=False,
            )
            if hasattr(res, "points"):
                current_points = [[int(round(p[0])), int(round(p[1]))] for p in res.points]
            stage_distances = [
                float(np.linalg.norm(np.asarray(p) - np.asarray(t)))
                for p, t in zip(current_points, stage_targets)
            ]
            final_distances = [
                float(np.linalg.norm(np.asarray(p) - np.asarray(t)))
                for p, t in zip(current_points, targets_yx)
            ]
            record = {
                "global_step": int(global_step),
                "stage": int(stage_index),
                "step_in_stage": int(step_in_stage),
                "target_fraction": float(stage_index / max(1, DRAG_NUM_STAGES)),
                "stage_mean_distance_px": float(np.mean(stage_distances)),
                "stage_max_distance_px": float(np.max(stage_distances)),
                "final_mean_distance_px": float(np.mean(final_distances)),
                "final_max_distance_px": float(np.max(final_distances)),
            }
            history.append(record)
            if step_in_stage % 10 == 0 or record["stage_max_distance_px"] <= DRAG_EARLY_STOP_PX or getattr(res, "stop", False):
                print(
                    f"  step {step_in_stage:03d} | stage mean={record['stage_mean_distance_px']:.3f} "
                    f"| stage max={record['stage_max_distance_px']:.3f} | final max={record['final_max_distance_px']:.3f}"
                )
            global_step += 1
            if record["stage_max_distance_px"] <= DRAG_EARLY_STOP_PX or getattr(res, "stop", False):
                break
    final_img = render_uint8_from_renderer(renderer, res)
    final_w = renderer.w.detach().cpu().numpy()
    return start_img, final_img, final_w, current_points, history


In [ ]:
drag_manifest = []

for idx, job in enumerate(inversion_jobs, start=1):
    pair_name = job["pair_name"]
    output_dir = OUTPUT_ROOT / pair_name
    output_dir.mkdir(parents=True, exist_ok=True)
    with open(PROJECT_ROOT / job["metadata_path"], "r", encoding="utf-8") as f:
        meta = json.load(f)
    w_plus = np.load(PROJECT_ROOT / meta["outputs"]["w_plus"])["w"]
    generator_pkl = resolve_generator_pkl(meta)
    handle_names, starts_yx, targets_yx, face_transform_meta = choose_handle_pairs(meta)

    print("\n" + "=" * 96)
    print(f"[03] True DragGAN latent edit :: {idx}/{len(inversion_jobs)} :: {pair_name}")
    print("Handle names:", handle_names)
    print("Generator pkl:", generator_pkl)
    print("Face target mode:", face_transform_meta["mode"])
    print("Driver keypoint:", face_transform_meta.get("driver_keypoint"))
    if face_transform_meta["mode"] in {"single_key_rigid_shift", "nose_drag_with_structure"}:
        print("Driver raw distance px:", round(face_transform_meta["driver_raw_distance_px"], 3))
        print("Driver delta capped:", face_transform_meta["driver_delta_capped"])
        print("Used delta xy:", [round(v, 3) for v in face_transform_meta["driver_used_delta_xy"]])
        if face_transform_meta["mode"] == "nose_drag_with_structure":
            print("Nose structure weights:", {
                "nose": face_transform_meta.get("nose_weight"),
                "chin": face_transform_meta.get("chin_weight"),
                "eye": face_transform_meta.get("eye_weight"),
                "ear_base": face_transform_meta.get("ear_base_weight"),
            })
            print("Structure affine blend:", face_transform_meta.get("structure_affine_blend"))
            print("Max support displacement px:", face_transform_meta.get("max_support_displacement_px"))
    else:
        print("Affine status:", face_transform_meta.get("affine_status"))
        print("Affine blend:", face_transform_meta.get("affine_blend"))
        print("Fit handles:", face_transform_meta.get("handle_names_used_for_fit"))
        if face_transform_meta["mode"] == "yaw_guided_affine":
            print("Turn sign x:", face_transform_meta.get("turn_sign_x"), "| source:", face_transform_meta.get("turn_source"))
            print("Yaw guide blend:", face_transform_meta.get("yaw_guide_blend"))
            print("Side weights:", face_transform_meta.get("side_weight_by_handle"))
        if face_transform_meta.get("affine_regularization"):
            reg = face_transform_meta["affine_regularization"]
            print("Affine singular values:", [round(v, 3) for v in reg.get("regularized_singular_values", [])])

    start_img, final_img, final_w, final_points_yx, history = run_draggan(w_plus, starts_yx, targets_yx, generator_pkl)
    np.savez_compressed(output_dir / "dragged_w_plus.npz", w=final_w)
    handle_plan = draw_handle_plan(start_img, starts_yx, targets_yx, handle_names)
    save_rgb(output_dir / "draggan_start_face_512.png", start_img)
    save_rgb(output_dir / "draggan_handle_plan_512.png", handle_plan)
    save_rgb(output_dir / "draggan_edited_face_512.png", final_img)

    drag_meta = {
        "pair_name": pair_name,
        "inversion_metadata_path": job["metadata_path"],
        "generator_pkl": str(generator_pkl),
        "handle_names": handle_names,
        "start_points_yx": starts_yx,
        "target_points_yx": targets_yx,
        "face_transform": face_transform_meta,
        "final_points_yx": final_points_yx,
        "drag_num_stages": int(DRAG_NUM_STAGES),
        "max_drag_steps_per_stage": int(MAX_DRAG_STEPS_PER_STAGE),
        "max_drag_steps": int(MAX_DRAG_STEPS),
        "drag_early_stop_px": float(DRAG_EARLY_STOP_PX),
        "drag_lr": float(DRAG_LR),
        "history": history,
        "outputs": {
            "draggan_start_face_512": str(output_dir / "draggan_start_face_512.png"),
            "draggan_handle_plan_512": str(output_dir / "draggan_handle_plan_512.png"),
            "draggan_edited_face_512": str(output_dir / "draggan_edited_face_512.png"),
            "dragged_w_plus": str(output_dir / "dragged_w_plus.npz"),
        },
    }
    with open(output_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(drag_meta, f, ensure_ascii=False, indent=2)

    show_images([
        ("GAN inversion start", start_img),
        ("Pose-preserving handle plan", handle_plan),
        ("True DragGAN edited face", final_img),
    ], cols=3, figsize=(15, 5))

    drag_manifest.append({
        "pair_name": pair_name,
        "metadata_path": str(output_dir / "metadata.json"),
        "output_dir": str(output_dir),
    })

with open(OUTPUT_ROOT / "batch_manifest.json", "w", encoding="utf-8") as f:
    json.dump(drag_manifest, f, ensure_ascii=False, indent=2)

print("Saved DragGAN edit manifest:", OUTPUT_ROOT / "batch_manifest.json")
